## Extended Thinking

**Extended thinking** lets Claude reason step by step before it answers. Claude writes its reasoning in a `thinking` block, followed by the final answer in a `text` block. It helps most on hard problems such as maths, planning, code and anything with strict rules.

### What changed in the latest models

Older models used a fixed *thinking budget*. Newer models such as `claude-haiku-5-5` use **adaptive thinking**: Claude decides for itself when and how much to think, and you steer it with `effort`.

| | Older models (for example `claude-haiku-4-5`) | Latest models (for example `claude-haiku-5-5`) |
|---|---|---|
| Turn thinking on | `thinking={"type": "enabled", "budget_tokens": 1024}` | On by default. Set `thinking={"type": "adaptive"}` to be explicit |
| How much to think | You set a token budget | Claude decides. You set `output_config={"effort": "low"}` to `"max"` |
| See the reasoning | The reasoning text is returned | It is empty unless you set `display` to `"summarized"` |
| `budget_tokens` | Required | Not accepted. It returns a 400 error |

```python
# Old (older models): a fixed budget of thinking tokens
thinking={"type": "enabled", "budget_tokens": 1024}

# New (latest models): adaptive thinking, a readable summary, and an effort level
thinking={"type": "adaptive", "display": "summarized"}
output_config={"effort": "high"}
```

In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

**What this cell does:** Loads your API key, creates the client, and picks the model.

- `load_dotenv()` reads `ANTHROPIC_API_KEY` from your `.env` file.
- `Anthropic()` finds the key automatically.
- `model = "claude-haiku-5-5"` is a latest-generation model, so it uses adaptive thinking.

## Helper Functions

The helpers below build a conversation and send it to Claude with adaptive thinking switched on.

In [5]:
# Helper functions
from anthropic.types import Message

# Magic string used in section 3 to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, tools=None, display="summarized", effort="medium"):
    params = {
        "model": model,
        # Thinking tokens count toward max_tokens, so leave plenty of room
        "max_tokens": 16000,
        "messages": messages,
        "thinking": {"type": "adaptive", "display": display},
        "output_config": {"effort": effort},
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

**What this cell does:** Defines the helpers used in the rest of the notebook.

- `add_user_message` and `add_assistant_message` add one message to the conversation. They accept either plain text or a whole response from Claude.
- `chat` sends the conversation to Claude. `"thinking": {"type": "adaptive", "display": display}` turns on adaptive thinking, and `"output_config": {"effort": effort}` sets how hard Claude thinks. There is no `budget_tokens`.
- `max_tokens` is `16000` because thinking counts toward that limit. A small limit can leave no room for the answer.
- `text_from_message` returns just the text blocks of a response, skipping the thinking.

## 1. Adaptive Thinking

**Definition:** With adaptive thinking, Claude decides whether a question needs reasoning and how much. A hard question gets more thinking, and a simple one gets little or none.

The response `content` is a **list of blocks**, in this order:

1. A `thinking` block with Claude's reasoning (readable because we set `display="summarized"`).
2. A `text` block with the final answer.

We loop over the blocks and check each block's `type`, because the thinking block comes first and has no `.text`.

In [6]:
messages = []
add_user_message(
    messages,
    "Create a haiku about Anthropic. Think carefully about syllable counts before answering.",
)

response = chat(messages, effort="high")

for block in response.content:
    if block.type == "thinking":
        print(f"Thinking: {block.thinking}")
    elif block.type == "text":
        print(f"Text: {block.text}")

Thinking: I'm working on a haiku about Anthropic, counting syllables for each line. The first line already fits with "Safe minds built with care," and I'm now starting on the second line, which needs seven syllables with "Anthropic" in it.

I'm testing a few seven-syllable options for the second line that name Anthropic, like "Anthropic guides AI's true path," which comes to seven.

I'm checking the syllable count for "Anthropic guides AI's true path" and finding that "AI" is two syllables, so that option comes to eight. I'm trying a few other seven-syllable phrasings for the second line that still name Anthropic.

I'm now testing line one options that avoid "safe" so the second line can keep "Anthropic seeks the safe path," which comes to seven syllables. I'm trying "Careful minds shape AI," which counts to six, and weighing a few other openings for the first line.

I'm settling the third line now: I need five syllables, and "Wise hands build AI" already fits that count. For the secon

**What this cell does:** Asks for a haiku and prints Claude's reasoning followed by the answer.

- The prompt asks Claude to think carefully about syllable counts, which a haiku needs.
- `effort="high"` tells Claude to think thoroughly. Use `low` for simple tasks and `high` for harder ones.
- The loop prints each block by type: `Thinking:` shows a summary of the reasoning, and `Text:` shows the haiku.

## 2. Controlling the Thinking Display

**Definition:** `display` controls whether you can *read* the reasoning. It does not change how much Claude thinks or what it costs.

| `display` | What you get in the `thinking` block |
|---|---|
| `"summarized"` | A readable summary of Claude's reasoning |
| `"omitted"` | An empty string. This is the default on the latest models |

> **Tip:** If you show reasoning to users and see nothing, `display` is probably `"omitted"`. Set it to `"summarized"`.

In [7]:
for display in ["summarized", "omitted"]:
    messages = []
    add_user_message(
        messages,
        "Create a haiku about Anthropic. Think carefully about syllable counts before answering.",
    )

    response = chat(messages, display=display, effort="high")

    print(f"--- display = {display} ---")
    for block in response.content:
        if block.type == "thinking":
            print(f"thinking block, {len(block.thinking)} characters of text")
        elif block.type == "text":
            print(f"text block: {block.text}")
    print()

--- display = summarized ---
thinking block, 591 characters of text
text block: Careful hands and minds (5)
Building AI, safe and true (7)
Claude learns to be kind (5)

--- display = omitted ---
thinking block, 0 characters of text
text block: Built with careful hands,
An-thro-pic shapes Claude's young mind,
Safety guides each thought.



**What this cell does:** Sends the same question twice and compares what the `thinking` block contains.

- With `display="summarized"`, the thinking block holds a summary, so its character count is above zero.
- With `display="omitted"`, the thinking block is still there but its text is empty (0 characters).
- `display` only changes what you can see. Claude thinks in the same way, and the answer is written the same way, in both cases.
- Claude chooses whether to think, so on a very easy question there may be no thinking block at all. This prompt asks it to think carefully, and `effort="high"` makes thinking likely.

## 3. Redacted Thinking

**Definition:** Anthropic's safety systems can occasionally flag part of Claude's reasoning and encrypt it. You then receive a `redacted_thinking` block instead of readable text. The content is still used internally, so Claude's answer is unaffected.

For testing, Anthropic provides a special magic string (`thinking_test_str`, defined in the helpers above). Sending it as the message makes the API return a redacted block, so you can check that your code handles one.

Your code should never assume that every thinking block is readable. Check each block's `type`.

In [8]:
messages = []
add_user_message(messages, thinking_test_str)

response = chat(messages)

# Print which kinds of block came back
print([block.type for block in response.content])

['redacted_thinking', 'text']


**What this cell does:** Sends the magic string and lists the type of every block in the response.

- Look for `redacted_thinking` in the list. If it appears, that block holds encrypted data instead of readable text, and a `text` block follows with the answer.
- If you see a plain `thinking` block instead, your code still works, because it checks `block.type` rather than assuming the first block is readable.
- When you send the conversation back to Claude (for example in a tool-use loop), pass the thinking blocks back unchanged. `add_assistant_message` does this when you give it the whole response.